# SA-DMAE Pre-training on Google Colab

**순서:**
1. GPU 확인 & Drive 마운트
2. 코드 클론 & 패키지 설치
3. 경로 설정
4. 데이터 확인
5. 학습 실행

> **런타임 → 런타임 유형 변경 → GPU (T4 or A100)** 먼저 설정할 것!

In [ ]:
# ── Cell 1: GPU 확인 & Drive 마운트 ──────────────────────────────────────────
import torch

print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM           :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('[!] GPU가 없습니다. 런타임 유형을 GPU로 변경하세요.')

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 2: 코드 클론 & 패키지 설치 ─────────────────────────────────────────
import os

REPO_DIR = '/content/SA-DMAE'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/whkim4338/SA-DMAE.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull   # 이미 있으면 최신 코드로 업데이트

%cd {REPO_DIR}

# timm / nibabel / tensorboard (torch, torchvision은 Colab 기본 설치)
!pip install timm nibabel tensorboard tqdm -q
print('패키지 설치 완료')

In [ ]:
# ── Cell 3: 경로 설정 (여기만 수정) ─────────────────────────────────────────

# .pt 파일 위치 (Drive 안에 있는 경로)
BRATS_PT  = '/content/drive/MyDrive/SA_DMAE_slices/brats'   # ← 실제 경로로 수정
UCSF_PT   = '/content/drive/MyDrive/SA_DMAE_slices/ucsf'    # ← 실제 경로로 수정

# 체크포인트 저장 위치 (Drive에 저장 → 세션 끊겨도 유지)
OUTPUT_DIR = '/content/drive/MyDrive/SA_DMAE_output'

# 학습 설정
EPOCHS      = 200
BATCH_SIZE  = 32    # T4: 16~32 / A100: 64 권장
ACCUM_ITER  = 2     # effective batch = BATCH_SIZE * ACCUM_ITER
SIGMA       = 0.25
MASK_RATIO  = 0.75
VAL_RATIO   = 0.1
SAVE_EVERY  = 10    # N epoch마다 체크포인트 저장

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 경로 확인
from pathlib import Path
brats_count = len(list(Path(BRATS_PT).glob('*.pt')))
ucsf_count  = len(list(Path(UCSF_PT).glob('*.pt')))
print(f'BraTS .pt : {brats_count}개')
print(f'UCSF  .pt : {ucsf_count}개')
print(f'합계      : {brats_count + ucsf_count}개')
print(f'Output    : {OUTPUT_DIR}')

In [ ]:
# ── Cell 4: 데이터 sanity check ──────────────────────────────────────────────
import torch, random
from pathlib import Path
import matplotlib.pyplot as plt

all_pts = list(Path(BRATS_PT).glob('*.pt')) + list(Path(UCSF_PT).glob('*.pt'))
sample  = torch.load(random.choice(all_pts), map_location='cpu')

print(f'샘플 shape : {list(sample.shape)}')   # (3, 3, 224, 224)
print(f'min={sample.min():.3f}  max={sample.max():.3f}')

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
fig.suptitle('Center slice — T1ce / T2 / FLAIR', fontsize=11)
for i, mod in enumerate(['T1ce', 'T2', 'FLAIR']):
    axes[i].imshow(sample[1, i].numpy(), cmap='gray', vmin=0, vmax=1)
    axes[i].set_title(mod); axes[i].axis('off')
plt.tight_layout()
plt.show()
print('데이터 확인 완료 — 학습 진행해도 됩니다.')

In [ ]:
# ── Cell 5: TensorBoard 실행 (학습 전 띄워두기) ───────────────────────────────
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}

In [ ]:
# ── Cell 6: 학습 실행 ─────────────────────────────────────────────────────────
# 이전 학습을 이어서 하려면 --resume 경로 지정
# (예: --resume /content/drive/MyDrive/SA_DMAE_output/checkpoint-50.pth)

!python main_pretrain_sa.py \
    --data_path  {BRATS_PT} \
    --ucsf_path  {UCSF_PT} \
    --preprocessed \
    --device     cuda \
    --model      sa_dmae_vit_base_patch16 \
    --epochs     {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --accum_iter {ACCUM_ITER} \
    --sigma      {SIGMA} \
    --mask_ratio {MASK_RATIO} \
    --val_ratio  {VAL_RATIO} \
    --save_every {SAVE_EVERY} \
    --output_dir {OUTPUT_DIR} \
    --log_dir    {OUTPUT_DIR} \
    --num_workers 2

In [ ]:
# ── Cell 7: 이어서 학습 (세션 끊겼을 때) ────────────────────────────────────
# 가장 최근 checkpoint 자동 탐색
import glob, os

ckpts = sorted(
    [f for f in glob.glob(f'{OUTPUT_DIR}/checkpoint-*.pth')
     if 'best' not in f],
    key=os.path.getmtime
)
if ckpts:
    latest = ckpts[-1]
    print(f'이어서 학습: {latest}')
    !python main_pretrain_sa.py \
        --data_path  {BRATS_PT} \
        --ucsf_path  {UCSF_PT} \
        --preprocessed \
        --device     cuda \
        --model      sa_dmae_vit_base_patch16 \
        --epochs     {EPOCHS} \
        --batch_size {BATCH_SIZE} \
        --accum_iter {ACCUM_ITER} \
        --sigma      {SIGMA} \
        --mask_ratio {MASK_RATIO} \
        --val_ratio  {VAL_RATIO} \
        --save_every {SAVE_EVERY} \
        --output_dir {OUTPUT_DIR} \
        --log_dir    {OUTPUT_DIR} \
        --resume     {latest} \
        --num_workers 2
else:
    print('저장된 체크포인트가 없습니다. Cell 6을 먼저 실행하세요.')

In [ ]:
# ── Cell 8: 학습 결과 요약 ────────────────────────────────────────────────────
import json
import matplotlib.pyplot as plt

log_path = f'{OUTPUT_DIR}/log.txt'
logs = [json.loads(l) for l in open(log_path)]

epochs     = [d['epoch'] + 1     for d in logs]
train_loss = [d['train_loss']     for d in logs]
val_loss   = [d['val_loss']       for d in logs]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(epochs, train_loss, label='Train Loss', linewidth=2)
ax.plot(epochs, val_loss,   label='Val Loss',   linewidth=2, linestyle='--')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('SA-DMAE Pre-training Loss')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/loss_curve.png', dpi=120)
plt.show()

best_epoch = min(range(len(val_loss)), key=lambda i: val_loss[i])
print(f'Best val loss : {val_loss[best_epoch]:.4f}  (epoch {epochs[best_epoch]})')
print(f'Final train   : {train_loss[-1]:.4f}')
print(f'Final val     : {val_loss[-1]:.4f}')